In [157]:
import sys, os

# Set project root (adjust the path as needed)
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [158]:
from research_and_analyst.utils.model_loader import ModelLoader
model_loader = ModelLoader()

{"timestamp": "2025-11-01T09:30:09.287554Z", "level": "info", "event": "GOOGLE_API_KEY loaded from environment"}
{"timestamp": "2025-11-01T09:30:09.299552Z", "level": "info", "event": "GROQ_API_KEY loaded from environment"}
{"timestamp": "2025-11-01T09:30:09.306553Z", "level": "info", "event": "ASTRA_DB_API_ENDPOINT loaded from environment"}


{"timestamp": "2025-11-01T09:30:09.312081Z", "level": "info", "event": "ASTRA_DB_APPLICATION_TOKEN loaded from environment"}
{"timestamp": "2025-11-01T09:30:09.316575Z", "level": "info", "event": "ASTRA_DB_KEYSPACE loaded from environment"}
{"timestamp": "2025-11-01T09:30:09.320580Z", "level": "info", "event": "CONFIG_PATH loaded from environment"}
{"config_keys": ["astra_db", "embedding_model", "retriever", "llm"], "timestamp": "2025-11-01T09:30:09.371922Z", "level": "info", "event": "YAML config loaded"}


In [159]:
llm = model_loader.load_llm()

{"provider": "google", "model": "gemini-2.0-flash", "timestamp": "2025-11-01T09:30:14.304031Z", "level": "info", "event": "Loading LLM"}


In [160]:
llm.invoke('hi').content

'Hi there! How can I help you today?'

In [161]:
from typing import List 
from typing_extensions import TypedDict 
from pydantic import BaseModel, Field 

from langgraph.graph import StateGraph, START, END 
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage 
from langgraph.checkpoint.memory import MemorySaver

# health

## Analyst( name="Dr. Neha Patel", role="Medical Data Scientist", affiliation="Stanford Medicine", description="Focuses on predictive models for patient outcomes." ),

## Analyst( name="Dr. Arun Verma", role="Ethics Researcher", affiliation="WHO", description="Explores ethical implications of AI in diagnostics." ), Analyst( name="Ms. Priya Sharma", role="Policy Analyst", affiliation="Ministry of Health", description="Investigates AI policy and compliance frameworks." )

In [162]:
class Analyst(BaseModel):
    name: str = Field(description="Name of the Analyst: ")
    role: str = Field(description="Role of the Analyst in the context of the topic: ")
    affiliation: str = Field(description="Primary Affiliation of the Analyst: ")
    description: str = Field(description="Description of the analyst focus, concerns, and motives")

    @property 
    def persona(self) -> str: 
         return f"Name: {self.name}\nRole: {self.role}\nAffiliation: {self.affiliation}\nDescription: {self.description}\n"

In [163]:
analyst = Analyst(
    name="Suel Ahmed",
    role="AI/ML Engineer",
    affiliation="AI Research LAB",
    description="I am genai developer as well as mentor"
    )

In [164]:
print(analyst.persona)

Name: Suel Ahmed
Role: AI/ML Engineer
Affiliation: AI Research LAB
Description: I am genai developer as well as mentor



In [165]:
class Perspectives(BaseModel):
       analysts: List[Analyst] = Field(description="Comprehensive list of analysts with their roles and affiliations.") 

In [166]:
class GenerateAnalystsState(TypedDict):
    topic: str #research topic
    max_analysts: int # number of analyst
    human_analyst_feedback: str # Human feedback
    analysts: List[Analyst] # Analyst asking questions

In [167]:
GenerateAnalystsState(
    topic='energy',
    max_analysts=5,
    human_analyst_feedback='give me real info'
)

{'topic': 'energy',
 'max_analysts': 5,
 'human_analyst_feedback': 'give me real info'}

In [168]:
Analyst(
        name="Dr. Neha Patel",
        role="Medical Data Scientist",
        affiliation="Stanford Medicine",
        description="Focuses on predictive models for patient outcomes."
    ),

(Analyst(name='Dr. Neha Patel', role='Medical Data Scientist', affiliation='Stanford Medicine', description='Focuses on predictive models for patient outcomes.'),)

In [169]:
analyst_instructions="""You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:

1. First, review the research topic:
{topic}
        
2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: 
        
{human_analyst_feedback}
    
3. Determine the most interesting themes based upon documents and / or feedback above.
                    
4. Pick the top {max_analysts} themes.

5. Assign one analyst to each theme."""

In [170]:
print([analyst_instructions.format(
        topic="education",
        max_analysts=4,
        human_analyst_feedback="please explain the recent trends in education improvement"
        
        )] + ["Generate the set of analysts."])

['You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:\n\n1. First, review the research topic:\neducation\n\n2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: \n\nplease explain the recent trends in education improvement\n\n3. Determine the most interesting themes based upon documents and / or feedback above.\n\n4. Pick the top 4 themes.\n\n5. Assign one analyst to each theme.', 'Generate the set of analysts.']


In [171]:
def create_analyst(state:GenerateAnalystsState):
    """
    it is creating my analyst
    
    """
    topic = state["topic"]
    max_analysts = state["max_analysts"]
    human_analyst_feedback = state.get("human_analyst_feedback","")
    
    structured_llm = llm.with_structured_output(Perspectives)
    
    system_messages = analyst_instructions.format(
        topic=topic,
        max_analysts=max_analysts,
        human_analyst_feedback=human_analyst_feedback
        
        )
    analysts = structured_llm.invoke([SystemMessage(content=system_messages)]+ [HumanMessage(content="Generate the set of analysts.")])
    
    # Write the list of analysis to state
    return {"analysts": analysts.analysts}
    

In [172]:
create_analyst(
    {'topic': 'education',
    'max_analysts': 4,
    'human_analyst_feedback': 'please explain the recent trends in education improvement'}
    )

{'analysts': [Analyst(name='Dr. Anya Sharma', role='Education Policy Advisor', affiliation='Center for Educational Equity', description='Focuses on equitable access to quality education, particularly for underserved communities. Concerned with systemic inequalities and advocates for policy changes that promote inclusivity and opportunity for all students.'),
  Analyst(name='Mr. Ben Carter', role='Educational Technology Specialist', affiliation='Global Tech Education', description='Interested in the integration of technology in education to enhance learning outcomes. Explores innovative digital tools and platforms, and their impact on student engagement and personalized learning experiences. Addresses concerns about digital literacy and equitable access to technology.'),
  Analyst(name='Ms. Chloe Davis', role='Curriculum Development Expert', affiliation='National Curriculum Board', description='Specializes in designing and evaluating curriculum frameworks that align with current educati

In [173]:
def human_feedback(state):
    """ No-op node that should be interrupted on """
    pass

def should_continue(state):
    """ Return the next node to execute """
    human_analyst_feedback = state.get("human_analyst_feedback",None)
    if human_analyst_feedback:
        return "create_analyst"

In [174]:
from IPython.display import Image, display
builder = StateGraph(GenerateAnalystsState)

builder.add_node("create_analyst",create_analyst)
builder.add_node("human_feedback", human_feedback)

In [175]:
builder.add_edge(START,"create_analyst")
builder.add_edge("create_analyst", "human_feedback")
builder.add_conditional_edges("human_feedback",
        should_continue,
        ["create_analyst",
        END])

In [183]:
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod

memory = MemorySaver()
graph = builder.compile(interrupt_before= ["human_feedback"],checkpointer= memory)
display(Image(graph.get_graph(xray=0).draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [142]:
topic = "Education"
max_analysts = 2
thread =  {"configurable":{"thread_id":1}}

for event in graph.stream({"topic":topic,
              "max_analysts":max_analysts},
             thread,
             stream_mode= "values"):
    analysts = event.get('analysts', '')
    
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50) 

Name: Dr. Anya Sharma
Affiliation: National Center for Fair Education
Role: Educational Equity Advocate
Description: Focuses on eliminating disparities in educational opportunities for marginalized communities, including students from low-income backgrounds and racial/ethnic minorities. Advocates for policy changes that promote equitable resource allocation and culturally responsive teaching.
--------------------------------------------------
Name: Professor Ben Carter
Affiliation: University of Innovative Learning
Role: Educational Technology Specialist
Description: An expert in integrating technology into education to enhance learning outcomes. Explores the use of AI, personalized learning platforms, and digital resources to create engaging and effective learning environments. Addresses concerns about digital equity and the ethical implications of technology in education.
--------------------------------------------------
Name: Ms. Chloe Davis
Affiliation: Inclusive Education Solutio

In [143]:
state = graph.get_state(thread)
print(state)

StateSnapshot(values={'topic': 'Education', 'max_analysts': 2, 'human_analyst_feedback': 'add something from more diverse perspectives, modern approaches and specially abled students education, do not overlook their unique challenges and needs also do not repeate the names', 'analysts': [Analyst(name='Dr. Amara Okoro', role='Special Education Consultant', affiliation='National Center for Learning Disabilities', description='Focuses on inclusive education practices and adaptive learning technologies for students with disabilities, advocating for personalized learning plans and accessible educational resources.'), Analyst(name='Professor Kenji Tanaka', role='Educational Technology Researcher', affiliation='University of Kyoto, Graduate School of Education', description='Specializes in modern approaches to education, particularly the integration of technology to enhance learning outcomes and engagement, with a focus on personalized learning and adaptive educational platforms.')]}, next=('

In [144]:
state.next

('human_feedback',)

In [145]:
graph.update_state(thread,
    {"human_analyst_feedback":"add something from more diverse perspectives, modern approaches and specially abled students education, do not overlook their unique challenges and needs also do not repeate the names"},as_node="human_feedback")

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0b49de-7b0c-693c-800a-9bf936998c8c'}}

In [146]:
for event in graph.stream(
    {"topic":topic,"max_analysts":max_analysts},
    thread,
    stream_mode= "values"):
    analysts = event.get('analysts', '')
    
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)  

Name: Dr. Amara Okoro
Affiliation: National Center for Learning Disabilities
Role: Special Education Consultant
Description: Focuses on inclusive education practices and adaptive learning technologies for students with disabilities, advocating for personalized learning plans and accessible educational resources.
--------------------------------------------------
Name: Professor Kenji Tanaka
Affiliation: University of Kyoto, Graduate School of Education
Role: Educational Technology Researcher
Description: Specializes in modern approaches to education, particularly the integration of technology to enhance learning outcomes and engagement, with a focus on personalized learning and adaptive educational platforms.
--------------------------------------------------
Name: Dr. Amara Okoro
Affiliation: National Center for Learning Disabilities
Role: Special Education Consultant
Description: Focuses on inclusive education practices and adaptive learning technologies for students with disabilitie

In [147]:
state = graph.get_state(thread)
state

StateSnapshot(values={'topic': 'Education', 'max_analysts': 2, 'human_analyst_feedback': 'add something from more diverse perspectives, modern approaches and specially abled students education, do not overlook their unique challenges and needs also do not repeate the names', 'analysts': [Analyst(name='Dr. Amara Okoro', role='Special Education Consultant', affiliation='National Center for Learning Disabilities', description='Focuses on inclusive education practices and adaptive learning technologies for students with disabilities, advocating for personalized learning plans and accessible educational resources.'), Analyst(name='Professor Kenji Tanaka', role='Educational Technology Researcher', affiliation='University of Kyoto, Graduate School of Education', description='Specializes in modern approaches to education, particularly the integration of technology to enhance learning outcomes and engagement, with a focus on personalized learning and adaptive educational platforms.')]}, next=('

In [148]:
state.next

('human_feedback',)

In [149]:

# If we are satisfied, then we simply supply no feedback
further_feedback = None
graph.update_state(thread, {"human_feedback":further_feedback}, as_node="human_feedback")

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0b49de-bf2c-6506-800e-68df49e3818f'}}

In [150]:
state = graph.get_state(thread)
state.next 

('create_analyst',)

In [151]:
final_state = graph.get_state(thread)
analysts = final_state.values.get('analysts')
analysts

[Analyst(name='Dr. Amara Okoro', role='Special Education Consultant', affiliation='National Center for Learning Disabilities', description='Focuses on inclusive education practices and adaptive learning technologies for students with disabilities, advocating for personalized learning plans and accessible educational resources.'),
 Analyst(name='Professor Kenji Tanaka', role='Educational Technology Researcher', affiliation='University of Kyoto, Graduate School of Education', description='Specializes in modern approaches to education, particularly the integration of technology to enhance learning outcomes and engagement, with a focus on personalized learning and adaptive educational platforms.')]

In [152]:
final_state.next

('create_analyst',)

1. Project Setup
Adds your project root to sys.path so you can import your own modules.
2. Model Loading
Imports and creates a ModelLoader object.
Loads a language model (llm) and tests it by sending a message ("hi").
3. Analyst Persona Classes
Defines an Analyst class (using Pydantic) to represent an AI analyst persona with name, role, affiliation, and description.
Creates an example analyst and prints their persona.
4. Analyst State and Instructions
Defines a Perspectives class to hold a list of analysts.
Defines a GenerateAnalystsState (using TypedDict) to represent the state for generating analysts.
Prepares instructions for generating analyst personas based on a research topic and feedback.
5. Analyst Generation Function
create_analyst function uses the LLM to generate a set of analyst personas based on the topic, feedback, and number of analysts requested.
6. LangGraph Workflow
Sets up a simple workflow (graph) using LangGraph:
Nodes: create_analyst, human_feedback
Edges: Control the flow between nodes based on feedback.
Visualizes the workflow graph.
7. Running the Workflow
Runs the workflow to generate analysts for a given topic.
Prints out the generated analysts’ details.
Allows for human feedback to refine the analysts and updates the workflow state.

## Second Workflow

In [153]:
import os 
from dotenv import load_dotenv
from langgraph.graph import MessagesState

load_dotenv()
tavily_api_key = os.getenv("TVLY_API_KEY")

from langchain_community.tools.tavily_search import TavilySearchResults

In [154]:
import urllib3

urllib3.disable_warnings()

In [155]:
import requests

response = requests.post("https://api.tavily.com/search", verify=False)
response 

<Response [401]>

In [156]:
# To install: pip install tavily-python
from tavily import TavilyClient

client = TavilyClient(tavily_api_key)
response = client.search(
    query="education"
)
print(response)

SSLError: HTTPSConnectionPool(host='api.tavily.com', port=443): Max retries exceeded with url: /search (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:992)')))

In [74]:
tavily_search = TavilySearchResults(
    tavily_api_key=tavily_api_key,
)

tavily_search.invoke("eduction")

'SSLError(MaxRetryError("HTTPSConnectionPool(host=\'api.tavily.com\', port=443): Max retries exceeded with url: /search (Caused by SSLError(SSLCertVerificationError(1, \'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:992)\')))"))'

In [ ]:
from langchain_community.document_loaders import WikipediaLoader 
from langchain_community.utilities import WikipediaAPIWrapper

wk = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=4000)
docs = wk.run("education")

print(docs)

In [ ]:
from typing import Annotated
import operator 

class InterviewState(MessagesState):
    max_num_turns: int # max number of turns
    context: Annotated[list, operator.add] # Source docs 
    analyst: Analyst # Analyst asking questions
    interview: str  # interview transcript
    sections: list # Final key we import 

class SearchQuery(BaseModel):
    search_query: str

In [ ]:
question_instructions = """You are an analyst tasked with interviewing an expert to learn about a specific topic. 

Your goal is boil down to interesting and specific insights related to your topic.

1. Interesting: Insights that people will find surprising or non-obvious.
        
2. Specific: Insights that avoid generalities and include specific examples from the expert.

Here is your topic of focus and set of goals: {goals}
        
Begin by introducing yourself using a name that fits your persona, and then ask your question.

Continue to ask questions to drill down and refine your understanding of the topic.
        
When you are satisfied with your understanding, complete the interview with: "Thank you so much for your help!"

Remember to stay in character throughout your response, reflecting the persona and goals provided to you."""

In [ ]:
interview_builder = StateGraph(InterviewState)

In [ ]:
print(question_instructions.format(goals = analyst.persona))

In [ ]:
print(analyst.persona)

In [ ]:
def generate_questions(state:InterviewState):
    """Node to geneate questions""" 
    analyst = state['analyst']
    message = state['messages']

    # call the llm 
    system_message = question_instructions.format(goals = analyst.persona)
    question = llm.invoke([SystemMessage(content=system_message)]+ message)
    return question

In [ ]:
state = {
    "max_num_turns":2,
    "analyst": analyst,
    "context":[],
    "interview":"",
    "section":[],
    "messages":[HumanMessage(content="Hi, I am your manager. Do the proper research based on your expertise by asking insightful questions?")]
}
print(state)

In [ ]:
print(generate_questions(state).content)

In [ ]:
from langchain_core.messages import get_buffer_string

# Search query writing
search_instructions = SystemMessage(content=f"""You will be given a conversation between an analyst and an expert. 
Your goal is to generate a well-structured query for use in retrieval and / or web-search related to the conversation. 
First, analyze the full conversation.
Pay particular attention to the final question posed by the analyst.
Convert this final question into a well-structured web search query""")

In [ ]:
def search_web(state:InterviewState):
    """
    Scrap the internet for information related to the interview question.
    """

    structured_llm = llm.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke([search_instructions]+ state['messages'])

    search_docs = tavily_search.invoke(search_query.search_query)
    
    # # Debugging
    #  print('*'*50)
    # print(search_docs)
    # print('*'*50)
    
    # Format
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
            for doc in search_docs
        ]
    )
    return {"context": [formatted_search_docs]}

def search_wikipedia(state:InterviewState):
    """
    Retrieve data from wiki
    """
    # Search query
    structured_llm = llm.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke([search_instructions]+state['messages'])
    
    print("*******************************")
    print(search_query)
    
    # Search
    search_docs = WikipediaLoader(query=search_query.search_query).load()

     # Format
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]}

answer_instructions = """You are an expert being interviewed by an analyst.

        Here is analyst area of focus: {goals}. 
                
        You goal is to answer a question posed by the interviewer.

        To answer question, use this context:
                
        {context}

        When answering questions, follow these guidelines:
                
        1. Use only the information provided in the context. 
                
        2. Do not introduce external information or make assumptions beyond what is explicitly stated in the context.

        3. The context contain sources at the topic of each individual document.

        4. Include these sources your answer next to any relevant statements. For example, for source # 1 use [1]. 

        5. List your sources in order at the bottom of your answer. [1] Source 1, [2] Source 2, etc
                
        6. If the source is: <Document source="assistant/docs/llama3_1.pdf" page="7"/>' then just list: 
                
        [1] assistant/docs/llama3_1.pdf, page 7 
                
        And skip the addition of the brackets as well as the Document source preamble in your citation."""

def generate_answer(state:InterviewState):
   
    """ Node to answer a question """

    # Get state
    analyst = state["analyst"]
    messages = state["messages"]
    context = state["context"]

    # Answer question
    system_message = answer_instructions.format(goals=analyst.persona, context=context)
    answer = llm.invoke([SystemMessage(content=system_message)]+messages)
            
    # Name the message as coming from the expert
    answer.name = "expert"
    
    # Append it to state
    return {"messages": [answer]}

def router_messages(state: InterviewState, 
                   name: str = "expert"):

    """ Route between question and answer """
    
    # Get messages
    messages = state["messages"]
    max_num_turns = state.get('max_num_turns',2)

    # Check the number of expert answers 
    num_responses = len(
        [m for m in messages if isinstance(m, AIMessage) and m.name == name]
    )

    # End if expert has answered more than the max turns
    if num_responses >= max_num_turns:
        return 'save_interview'

    # This router is run after each question - answer pair 
    # Get the last question asked to check if it signals the end of discussion
    last_question = messages[-2]
    
    if "Thank you so much for your help" in last_question.content:
        return 'save_interview'
    
    return "ask_question"

def save_interview(state: InterviewState):
    
    """ Save interviews """

    # Get messages
    messages = state["messages"]
    
    # Convert interview to a string
    interview = get_buffer_string(messages)
    
    # Save to interviews key
    return {"interview": interview}

section_writer_instructions = """You are an expert technical writer. 
            
Your task is to create a short, easily digestible section of a report based on a set of source documents.

1. Analyze the content of the source documents: 
- The name of each source document is at the start of the document, with the <Document tag.
        
2. Create a report structure using markdown formatting:
- Use ## for the section title
- Use ### for sub-section headers
        
3. Write the report following this structure:
a. Title (## header)
b. Summary (### header)
c. Sources (### header)

4. Make your title engaging based upon the focus area of the analyst: 
{focus}

5. For the summary section:
- Set up summary with general background / context related to the focus area of the analyst
- Emphasize what is novel, interesting, or surprising about insights gathered from the interview
- Create a numbered list of source documents, as you use them
- Do not mention the names of interviewers or experts
- Aim for approximately 400 words maximum
- Use numbered sources in your report (e.g., [1], [2]) based on information from source documents
        
6. In the Sources section:
- Include all sources used in your report
- Provide full links to relevant websites or specific document paths
- Separate each source by a newline. Use two spaces at the end of each line to create a newline in Markdown.
- It will look like:

### Sources
[1] Link or Document name
[2] Link or Document name

7. Be sure to combine sources. For example this is not correct:

[3] https://ai.meta.com/blog/meta-llama-3-1/
[4] https://ai.meta.com/blog/meta-llama-3-1/

There should be no redundant sources. It should simply be:

[3] https://ai.meta.com/blog/meta-llama-3-1/
        
8. Final review:
- Ensure the report follows the required structure
- Include no preamble before the title of the report
- Check that all guidelines have been followed"""

def write_section(state: InterviewState):

    """ Node to answer a question """

    # Get state
    interview = state["interview"]
    context = state["context"]
    analyst = state["analyst"]
   
    # Write section using either the gathered source docs from interview (context) or the interview itself (interview)
    system_message = section_writer_instructions.format(focus=analyst.description)
    section = llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content=f"Use this source to write your section: {context}")]) 
                
    # Append it to state
    return {"sections": [section.content]}

In [ ]:
# state for search web 
txt = """  
Okay, understood. Hello Mr. Tanaka, my name is Anya Sharma, and I'm an analyst focusing on emerging trends in EdTech. I'm hoping to tap into your expertise as a Technology Integration Specialist at Global EdTech Solutions.
To start, I'm interested in understanding where you see the *most* impactful, yet perhaps *underutilized*, opportunities for technology integration in education right now. 
We hear a lot about AI and personalized learning, but I'm curious about what's flying under the radar.
Could you share a specific example of a technology or approach that you think deserves more attention and why?"""

state = {
    "max_num_turns":4,
    "analyst": analyst,
    "context":[],
    "interview":"",
    "section":[],
    "messages":[AIMessage(content=txt)]
}

result = search_web(state)

print(result['context'][0])

In [ ]:
state = {
    "max_num_turns":4,
    "analyst": analyst,
    "context":[],
    "interview":"",
    "section":[],
    "messages":[AIMessage(content=txt)]
}

result = search_wikipedia(state)

print(result)

In [ ]:
# Add nodes to the graph
interview_builder.add_node("ask_question", generate_questions)
interview_builder.add_node("search_web", search_web)
interview_builder.add_node("search_wikipedia", search_wikipedia)
interview_builder.add_node("generate_answer", generate_answer)
interview_builder.add_node("save_interview", save_interview)
interview_builder.add_node("write_section", write_section)

# Define edges between nodes
interview_builder.add_edge(START, "ask_question") 
interview_builder.add_edge("ask_question", "search_web") 
interview_builder.add_edge("ask_question", "search_wikipedia") 
interview_builder.add_edge("search_web","generate_answer") 
interview_builder.add_edge("search_wikipedia","generate_answer") 
interview_builder.add_conditional_edges("generate_answer",
                           router_messages,["ask_question","save_interview"]) 
interview_builder.add_edge("save_interview", "write_section")
interview_builder.add_edge("write_section", END)

In [ ]:
interview_graph = interview_builder.compile(checkpointer= memory).with_config(run_name = "interview-conducter")

In [ ]:
display(Image(interview_graph.get_graph().draw_mermaid_png()))

In [ ]:
print(analyst.persona)

In [ ]:
# The interview_graph is a workflow built using LangGraph's StateGraph.
# It automates an interview process between an analyst and an expert, using an LLM to generate questions, search for information, answer, and write a report section.

# Here's how it works:
# 1. Starts with the analyst asking a question (ask_question node).
# 2. Searches for relevant information on the web or Wikipedia (search_web or search_wikipedia nodes).
# 3. Uses the gathered context to generate an expert answer (generate_answer node).
# 4. Routes between asking more questions or saving the interview, based on the number of turns or if the interview is complete (router_messages).
# 5. Saves the interview transcript (save_interview node).
# 6. Writes a summary section/report based on the interview and sources (write_section node).
# 7. Ends the workflow.

# The graph manages state and transitions automatically, allowing for multi-turn interviews and structured outputs.

In [ ]:
from IPython.display import Markdown

thread = {"configurable": {"thread_id": "1"}}
messages = [HumanMessage(content="education?")]
interview = interview_graph.invoke({"analyst":analyst,"messages":messages,"max_num_turns":2},thread)

In [ ]:
Markdown(interview['sections'][0])

In [ ]:
""" 

def search_wikipedia(state: InterviewState):
    # Use the LLM to rewrite the query to a concise topic for Wikipedia
    rewrite_prompt = SystemMessage(
        content="Given the following research question or query, extract the main topic or phrase that would best match a Wikipedia article title. Respond with only the topic or phrase."
    )
    user_message = state['message'][-1]  # Get the latest message (AIMessage or HumanMessage)
    concise_query = llm.invoke([rewrite_prompt, user_message]).content.strip()

    print("Original query:", user_message.content)
    print("Rewritten for Wikipedia:", concise_query)

    # Now use the concise query for Wikipedia search
    search_docs = WikipediaLoader(query=concise_query).load()
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}"/>\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )
    return {"context": [formatted_search_docs]}
"""

## Third Workflow

In [ ]:
from typing_extensions import TypedDict

class ResearchGrapthState(TypedDict):
    topic: str
    max_analysts: int
    human_analyst_feedback: str
    analysts: List[Analyst]
    sections: Annotated[list, operator.add]
    introductions: str 
    content: str 
    conclusion: str 
    final_report: str

In [ ]:
from langgraph.types import Send 

In [ ]:
def initiate_all_interviews(state:ResearchGrapthState):
    """ Initiate all interviews for all analysts """
    human_analyst_feedback = state.get('human_analyst_feedback')
    if human_analyst_feedback:
        return "create_analyst"
    
    else: 
        topic = state['topic'] 
        return [Send(
            'conduct_interview',{
                'analyst': analyst,
                'messages':[HumanMessage(content=f'So you said you are writing an article on {topic}?')]
            }
        ) for analyst in state['analysts']]

In [ ]:
report_writer_instructions = """You are a technical writer creating a report on this overall topic: 

{topic}
    
You have a team of analysts. Each analyst has done two things: 

1. They conducted an interview with an expert on a specific sub-topic.
2. They write up their finding into a memo.

Your task: 

1. You will be given a collection of memos from your analysts.
2. Think carefully about the insights from each memo.
3. Consolidate these into a crisp overall summary that ties together the central ideas from all of the memos. 
4. Summarize the central points in each memo into a cohesive single narrative.

To format your report:
 
1. Use markdown formatting. 
2. Include no pre-amble for the report.
3. Use no sub-heading. 
4. Start your report with a single title header: ## Insights
5. Do not mention any analyst names in your report.
6. Preserve any citations in the memos, which will be annotated in brackets, for example [1] or [2].
7. Create a final, consolidated list of sources and add to a Sources section with the `## Sources` header.
8. List your sources in order and do not repeat.

[1] Source 1
[2] Source 2

Here are the memos from your analysts to build your report from: 

{context}"""

In [ ]:
def write_report_old(state:ResearchGrapthState):
    """ Write the final report """
    sections = state['sections'] 
    topic = state['topic'] 

    formatted_str_sections = "\n\n---\n\n".join([f'{section}' for section in sections])

    system_message = report_writer_instructions.format(topic = topic, context = formatted_str_sections)
    report = llm.invoke([SystemMessage(content=system_message)] + [HumanMessage(content="Write a report based upon these memos.")])
    return {'content': report.content}

def write_report(state: ResearchGrapthState):
    # Full set of sections
    sections = state["sections"]
    topic = state["topic"]

    # Concat all sections together
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])
    
    # Summarize the sections into a final report
    system_message = report_writer_instructions.format(topic=topic, context=formatted_str_sections)    
    report = llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content=f"Write a report based upon these memos.")]) 
    return {"content": report.content}

In [ ]:
intro_conclusion_instructions = """You are a technical writer finishing a report on {topic}

You will be given all of the sections of the report.

You job is to write a crisp and compelling introduction or conclusion section.

The user will instruct you whether to write the introduction or conclusion.

Include no pre-amble for either section.

Target around 100 words, crisply previewing (for introduction) or recapping (for conclusion) all of the sections of the report.

Use markdown formatting. 

For your introduction, create a compelling title and use the # header for the title.

For your introduction, use ## Introduction as the section header. 

For your conclusion, use ## Conclusion as the section header.

Here are the sections to reflect on for writing: {formatted_str_sections}"""

In [ ]:
def write_introduction(state: ResearchGrapthState):
    # Full set of sections
    sections = state["sections"]
    topic = state["topic"]

    # Concat all sections together
    formatted_str_sections = "\n\n".join([f"{section}" for section in sections])
    
    # Summarize the sections into a final report
    
    instructions = intro_conclusion_instructions.format(topic=topic, formatted_str_sections=formatted_str_sections)    
    intro = llm.invoke([instructions]+[HumanMessage(content=f"Write the report introduction")]) 
    return {"introduction": intro.content}

def write_conclusions(state: ResearchGrapthState):
    sections = state['sections'] 
    topic = state['topic'] 

    formatted_str_sections = "\n\n---\n\n".join([f'{section}' for section in sections])

    instructions = intro_conclusion_instructions.format(topic = topic, formatted_str_sections = formatted_str_sections)
    report = llm.invoke([SystemMessage(content=instructions)] + [HumanMessage(content="Write the report conclusion.")])
    return {'content': report.content}

In [ ]:
def finalize_report(state:ResearchGrapthState):
    """ Finalize the report with intro and conclusion """
    # Save full final report
    content = state["content"]
    if content.startswith("## Insights"):
        content = content.strip("## Insights")
        
    if "## Sources" in content:
        try:
            content, sources = content.split("\n## Sources\n")
        except:
            sources = None
    else:
        sources = None

    final_report = state["introduction"] + "\n\n---\n\n" + content + "\n\n---\n\n" + state["conclusion"]
    if sources is not None:
        final_report += "\n\n## Sources\n" + sources
    return {"final_report": final_report}

In [ ]:
# Add nodes and edges 
builder = StateGraph(ResearchGrapthState)
builder.add_node("create_analyst", create_analyst)
builder.add_node("human_feedback", human_feedback)
builder.add_node("conduct_interview", interview_builder.compile())
builder.add_node("write_report",write_report)
builder.add_node("write_introduction",write_introduction)
builder.add_node("write_conclusion",write_conclusions)
builder.add_node("finalize_report",finalize_report)

# Logic
builder.add_edge(START, "create_analyst")
builder.add_edge("create_analyst", "human_feedback")
builder.add_conditional_edges("human_feedback", initiate_all_interviews, ["create_analyst", "conduct_interview"])
builder.add_edge("conduct_interview", "write_report")
builder.add_edge("conduct_interview", "write_introduction")
builder.add_edge("conduct_interview", "write_conclusion")
builder.add_edge(["write_conclusion", "write_report", "write_introduction"], "finalize_report")
builder.add_edge("finalize_report", END)

In [ ]:
memory = MemorySaver()
graph = builder.compile(interrupt_before=['human_feedback'], checkpointer=memory)

In [ ]:
display(Image(graph.get_graph(xray=1).draw_mermaid_png(draw_method='pyppeteer')))

In [ ]:
max_analysts = 3
topic = "How can generative help us to play the cricket?"
thread = {"configurable": {"thread_id": "1"}}

In [ ]:
# Run the graph until the first interview
for event in graph.stream({"topic":topic,"max_analysts":max_analysts}, thread, stream_mode="values"):
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)

In [ ]:
graph.update_state(thread, {"human_analyst_feedback":"please include more insights on how generative ai can help indian cricket team to improve their performance"},as_node="human_feedback")

In [ ]:
# Run the graph until the first interruption
for event in graph.stream({"topic":topic,"max_analysts":max_analysts}, thread, stream_mode="values"):
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50) 

In [ ]:
graph.update_state(thread, {"human_analyst_feedback":""}, as_node="human_feedback")

In [ ]:
graph.get_state(thread).next

In [ ]:
# Continue
for event in graph.stream(None, thread, stream_mode="updates"):
    print("--Node--")
    node_name = next(iter(event.keys()))
    print(node_name)
    break # Final report 

In [ ]:
from IPython.display import Markdown
final_state = graph.get_state(thread)
report = final_state.values.get('final_report')
Markdown(report)